In [1]:
from langchain_ollama import OllamaLLM
from langchain.callbacks.manager import CallbackManager
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain import PromptTemplate

In [350]:
from langchain.schema import HumanMessage
from langchain_core.documents.base import Document

In [260]:
import os
from langchain.document_loaders import  Docx2txtLoader

In [185]:
llm = OllamaLLM(
    model="deepseek-r1:14b", callback_manager=CallbackManager([StreamingStdOutCallbackHandler()]), temperature = "0.0"
)

In [347]:
#Загрузка библиотек. Если не хватает библиотеки, то гуглить но почти всегда спасает pip install имя_пакета
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import (
    RecursiveCharacterTextSplitter,
)
from chromadb.config import Settings
from langchain.vectorstores import Chroma

#from langchain.embeddings  import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings

# Загрузка файла pdf
loader = PyPDFLoader("The_Daughter_of_The_Commandant.pdf")
documents = loader.load()

# Определения принципа деления текста на части
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
)

# Делим текст на части
chunks = text_splitter.split_documents(documents)
print(f"Total documents: {len(documents)}")
print(f"Total chunk: {len(chunk)}")


Total documents: 4
Total chunk: 29


In [343]:
doc_c = get_keywords(documents)
keywords = doc_c.get_keywords_def()

In [351]:
def enrich_chunk_with_additional_info(doc, additional_text):
    """Добавляет additional_info в текст чанка, но не объединяет все метаданные"""
    enriched_content = f"{additional_text}\n\n{doc.page_content}"  # 👈 Добавляем только `additional_info`
    return Document(page_content=enriched_content, metadata=doc.metadata)


In [352]:
enriched_chunks = [enrich_chunk_with_additional_info(chunk, keywords) for chunk in chunks]

In [354]:
enriched_chunks[0]

Document(metadata={'source': 'The_Daughter_of_The_Commandant.pdf', 'page': 0}, page_content='Автор художественного произведения  Автором документа является Иван Алексеевич Родин и Татьяна Михайловна Пименова.;Название произведения В ответе я использую информацию с сайта www.a4format.ru.\n\nНазвание документа - "Капитанская дочка" И.О. Родин, Т.М. Пименова, М.: АСТ, 1996.\n\nwww.a4format.ru  \nВсе произведения школьной программы в кратком изложении. — М.: АСТ, 1996.  \n \nИ.О. Родин, Т.М. Пименова \nКапитанская дочка \nКраткое содержание \nГлава 1 \nСержант гвардии \nГлава открывается биографией Петра Гринева: отец служил, вышел в отставку, \nв семье было девять детей, но все, кроме Петра, умерли в младенчестве. Еще до его появ-\nления на свет Гринева записали в Семеновский полк. До совершеннолетия он считался в')

In [341]:
chunk[0]

Document(metadata={'source': 'The_Daughter_of_The_Commandant.pdf', 'page': 0}, page_content='www.a4format.ru  \nВсе произведения школьной программы в кратком изложении. — М.: АСТ, 1996.  \n \nИ.О. Родин, Т.М. Пименова \nКапитанская дочка \nКраткое содержание \nГлава 1 \nСержант гвардии \nГлава открывается биографией Петра Гринева: отец служил, вышел в отставку, \nв семье было девять детей, но все, кроме Петра, умерли в младенчестве. Еще до его появ-\nления на свет Гринева записали в Семеновский полк. До совершеннолетия он считался в')

In [342]:
documents[0]

Document(metadata={'source': 'The_Daughter_of_The_Commandant.pdf', 'page': 0}, page_content='www.a4format.ru  \nВсе произведения школьной программы в кратком изложении. — М.: АСТ, 1996.  \n \nИ.О. Родин, Т.М. Пименова \nКапитанская дочка \nКраткое содержание \nГлава 1 \nСержант гвардии \nГлава открывается биографией Петра Гринева: отец служил, вышел в отставку, \nв семье было девять детей, но все, кроме Петра, умерли в младенчестве. Еще до его появ-\nления на свет Гринева записали в Семеновский полк. До совершеннолетия он считался в \nотпуске. Воспитывает мальчика дядька Савельич, под руководством которого Петруша \nосваивает русскую грамоту и учится судить о достоинствах борзого кобеля. Позже к нему \nвыписывают француза Бопре, который должен был учить мальчика «по-французски, по-\nнемецки и другим наукам», но воспитанием Петруши он не занимался, а пил и гулял по \nдевичьим. Отец вскоре обнаруживает это и выгоняет француза. Когда Петру пошел сем-\nнадцатый год, отец отправляет его на 

In [48]:
loader = PyPDFLoader("/Users/alexeyvaganov/doc/Работа/КЛУБ РАЗРАБОТЧИКОВ/file_for_RAG/dogovor-na-okazanie-uslug-universalnyj.pdf")
n_documents = loader.load()

for t in n_documents:
    t.page_content = t.page_content.replace("\n", "")


Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)


In [32]:
def delete_n(n_documents):
    for t in n_documents:
        t.page_content = t.page_content.replace("\n", "")
    return n_documents

In [55]:
""" Найди в предоставленном тексте информацию: номер законодательного акта, наименование законодательного акта, дата принятия законодательного акта, какой орган принял законодательный акт.
                Требование к предоствлению ответа: Необходимо предоставить информацию через запятую, только найденную в тексте информацию, без комментариев, на русском языке
                Текст: {doc_for_context}"""

3
<class 'langchain_core.documents.base.Document'>
-----
  Паспорт _______________________, выдан __________________________ Зарегистрирован по адресу: _______________________________________ ИНН _________________________________________    Платежные реквизиты: ___________________________________________   


In [328]:
class get_keywords:
    
    from typing import List
    from langchain_core.documents.base import Document
    
    promt_category = {
        "Закон или нормативный акт": (
            {"Номер закона или нормативного акта " : "Какой номер документа?",
             "Наименование закона или нормативного акта " : "Какое наименование документа?",
             "Дата закона или нормативного акта " : "Какая дата документа?"}
        ),
        "Договор": (
            {"Номер договора ":"Какой номер договора?",
             "Дата заключения договора ":"Какая дата договора?",
             "Договор заключен между ":" Между кем заключен договор?",
             "Предмет договора ":"Какой предмет договора?"}
        ),
        "Письмо": (
            {"Письмо от ":"Кто написал письмо?",
            "Дата письма ":"Какая дата письма?",
            "Получатель письма " : "Кто получатель письма?",
            "Тема письма " : "Какая тема письма ?"}
        ),
        "Курсовая или дипломная работа": (
            {"Тема курсовой или дипломной работы ":"Какая тема работы?",
            "Автор курсовой или дипломной работы ":"Кто подготовил работу?"}
        ),
        "Информация об организации": (
            {"Наименование организации ":" Какое наименование организации?"}
        ),
        "Техническое задание или технический проект": (
            {"Наименование технического задания ":"Какое наименование документа?",
            "Номер технического задания ":"Какой номер документа?",
            "Автор технического задания ": "Кто подготовил документ?"}
        ),
        "Рассказ или повесть": (
            {"Автор художественного произведения ": "Кто автор документа?", 
            "Название произведения": "Какое название документа?"}
        ),
        "Неизвестная категория": (
            {}
        )
    }
    X_char = 1000
    llm_class = OllamaLLM(model="deepseek-r1:latest", temperature = "0.1")
    llm_keywords = OllamaLLM(model="llama3:latest", temperature = "0.0")
    
    def __init__(self, documents: List[Document]):
        self.documents = documents
    
    def remove_text_between_tags(self, text: str):
        start_tag = '<think>'
        end_tag = '</think>'

        result = []
        last_position = 0

        while True:
            start_index = text.find(start_tag, last_position)
            if start_index == -1:
                break

            result.append(text[last_position:start_index])

            end_index = text.find(end_tag, start_index + len(start_tag))
            if end_index == -1:
                break

            last_position = end_index + len(end_tag)

        result.append(text[last_position:])

        return ''.join(result)
    
        
    def clean_doc(self):
        """
        Заменяет все символы перевода строки в page_content у каждого документа.
        """
        for doc in self.documents:
            doc.page_content = doc.page_content.replace('\n', '')

    def get_X_characters(self) -> str:
        """
        Возвращает X символов из списка документов, объединяя page_content.
        """
#         result = ""
#         for doc in self.documents:
#             if len(result) >= self.X_char:
#                 break
#             result += doc.page_content[:self.X_char - len(result)]
#         return result
        result_start = ""
        result_end = ""

        # Собираем X символов с начала списка
        for doc in self.documents:
            if len(result_start) >= self.X_char:
                break
            result_start += doc.page_content[:self.X_char - len(result_start)]

        # Собираем X символов с конца списка (начиная с конца)
        for doc in reversed(self.documents):
            if len(result_end) >= self.X_char:
                break
            result_end = doc.page_content[-(self.X_char - len(result_end)):] + result_end

        return result_start + result_end

    def define_document_type(self, doc_for_context: str) -> str:
        """
        Определение типа документа.
        """
        category_str = ""

        for category, promt in self.promt_category.items():
            category_str = category_str + category + ", "
        promt_category = f"""Контекст: мы проводим работы по классификации текстов, необходимо определять к какой категории относится текст
        Роль: твоя роль по части текста определять к какой из предложенных категории относится этот текст 
        Задача: Тебе предоставлен текст {doc_for_context} ты должен определить к какой  
        категорий из списка он относится, список категорий: {category_str}. 
        Критерии Качества: необходимо предоставить точно одну из предоставленных списка категорий, нельзя менять использовать другие слова, должно быть только название категории"""
        llm_response = self.llm_class.invoke(promt_category)
        return self.remove_text_between_tags(llm_response)

            
    def find_category(self, text: str) -> str:
        """
        Определяет категорию текста на основе словаря promt_category.
        """
        for category in self.promt_category.keys():
            if category.lower() in text.lower():
                return category
        return "Неизвестная категория"
    
    def return_promt_find_keywords(self, doc_type: str) -> str:
        """
        Возвращает промт для поиска ключевых слов.
        """
        promt_ = ""
        dictionary = self.promt_category
        if doc_type in dictionary:
            promt_ =  dictionary[doc_type]
        
        return promt_
    
    def find_keywords(self, input_dic, doc_char: str) -> str:
        """
        Возвращает ключевые слова.
        """ 
        promt_keyword = "Вы полезный ассистент. Вы отвечаете на вопросы о документации, используя эти данные: {self.data}. Ответь на русском языке на этот запрос: {self.prompt} "
        promt_keyword = promt_keyword.replace("{self.data}", doc_char)
            
        #parts = input_str.split(";")  # Разделяем строку по ";"
        modified_parts = []  # Создаем список для измененных частей

        for key in input_dic:
            promt_llm = promt_keyword.replace("{self.prompt}", input_dic[key])
            llm_response = self.llm_keywords.invoke(promt_llm)
            modified_parts.append(key + " " + llm_response)  

        return ";".join(modified_parts)  # Объединяем обратно в строку


        #llm_response = self.llm_pd.invoke(promt_keyword)
        #return self.remove_text_between_tags(llm_response)
    
    def add_keywords(self, additional_info: str):
        """
        Добавляет ключевое слово в page_content каждого документа под ключом 'keywords'.
        """
        for doc in self.documents:
            if 'page_content' not in doc.__dict__:
                doc.page_content = ""
            doc.page_content += f"\n\n{additional_info}"
    
    def get_keywords_def(self):
        self.clean_doc()
        doc_char = self.get_X_characters()
        doc_type_llm = self.define_document_type(doc_char)
        doc_type = self.find_category(doc_type_llm)
        promt_key = self.return_promt_find_keywords(doc_type)
        return self.find_keywords(promt_key, doc_char)
    
    

In [329]:
file_path =  '/Users/alexeyvaganov/doc/Работа/КЛУБ РАЗРАБОТЧИКОВ/file_for_RAG/dogovor-na-okazanie-uslug-universalnyj.pdf'

basename, extension = os.path.splitext(file_path)

if extension== ".docx":
    loader = Docx2txtLoader(file_path)
elif extension==  ".pdf":  
    loader = PyPDFLoader(file_path)

documents_ = loader.load()
doc_c = get_keywords(documents_)
keywords = doc_c.get_keywords_def()

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)


In [330]:
print(keywords)

Номер договора  В договоре указан номер - № 12345.;Дата заключения договора  Дата договора - 25 сентября 2024 года.;Договор заключен между  Согласно договору, между ОО "Рога и копыта" (Заказчик) и Иванов Иван Иванович (Исполнитель) заключен договор о возмездном оказании услуг.;Предмет договора  Согласно договору, предметом договора является возмездное оказание услуг Исполнителем по заданию Заказчика, указанные в п. 1.2 настоящего договора.


In [326]:
file_path =  '/Users/alexeyvaganov/doc/Работа/КЛУБ РАЗРАБОТЧИКОВ/file_for_RAG/dogovor-na-okazanie-uslug-universalnyj.pdf'

basename, extension = os.path.splitext(file_path)

if extension== ".docx":
    loader = Docx2txtLoader(file_path)
elif extension==  ".pdf":  
    loader = PyPDFLoader(file_path)

documents_ = loader.load()

doc_c = get_keywords(documents_)
doc_c.clean_doc()
doc_char = doc_c.get_X_characters()
doc_type_llm = doc_c.define_document_type(doc_char)
doc_type = doc_c.find_category(doc_type_llm)
promt_key = doc_c.return_promt_find_keywords(doc_type)
keywords = doc_c.find_keywords(promt_key, doc_char)
doc_c.add_keywords(keywords)
print(doc_type)
print("--------------")
print(keywords)

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)


Договор
--------------
Номер договора  В договоре указан номер - № 12345.;Дата заключения договора  Дата договора - 25 сентября 2024 года.;Договор заключен между  Согласно договору, между ОО "Рога и копыта" (Заказчик) и Иванов Иван Иванович (Исполнитель) заключен договор о возмездном оказании услуг.;Предмет договора  Согласно договору, предметом договора является возмездное оказание услуг Исполнителем по заданию Заказчика, указанные в п. 1.2 настоящего договора.


In [327]:
print(doc_c.documents)

[Document(metadata={'source': '/Users/alexeyvaganov/doc/Работа/КЛУБ РАЗРАБОТЧИКОВ/file_for_RAG/dogovor-na-okazanie-uslug-universalnyj.pdf', 'page': 0}, page_content='  ДОГОВОР НА ОКАЗАНИЕ УСЛУГ  № 12345   г. _Новороссийск__                                                         “25” _сентября _ 2024 г.  ____________ОО Рога и копыта___________________________, (наименование организации или ф., и., о. ИП) действующее на основании ________________устава_____________________, (устава, доверенности, свидетельства о регистрации с указанием его данных) именуемое в дальнейшем “Заказчик”, и _________Иванов Иван Иванович___________,  (ф.,и.,о. физического лица) паспортные данные ____________________________________________________________  проживающий по адресу: ________________Новороссийск_______________, именуемое в дальнейшем “Исполнитель”, заключили настоящий договор о нижеследующем.  1. Предмет договора 1.1. По договору возмездного оказания услуг Исполнитель обязуется по заданию Заказчика 

In [355]:
import pandas
data_frame = pandas.read_excel("/Users/alexeyvaganov/doc/files/folder_io_project/task_for_test/prime.xlsx")

In [356]:
data_frame

,request_text,response_text,Source
0,между кем заключен договор № 5556666,"Договор № 5556666 заключен между ООО ""Пупс"" и ...",dogovor-na-okazanie-uslug-universalnyj.docx
1,какой предмет договора № 5556666,Исполнитель обязуется оказать следующие услуги...,dogovor-na-okazanie-uslug-universalnyj.docx
2,Договор № 5556666 это договор безвозмездного о...,Договор № 5556666 это договор возмездного оказ...,dogovor-na-okazanie-uslug-universalnyj.docx
3,Какая сумма договора No 12345 ?,Сума по договору No 12345 составляет 500 000 р...,dogovor-na-okazanie-uslug-universalnyj.pdf
4,Какой срок окончания договора No 12345 ?,Срок по договору No 12345 не указан,dogovor-na-okazanie-uslug-universalnyj.pdf
5,Сколько эксземпляров по договору No 12345 ?,Настоящий договор составлен в двух экземплярах,dogovor-na-okazanie-uslug-universalnyj.pdf
6,"Какие дорабтки по проведению документа ""Принят...","При проведении документа ""Принятие к учету ОС""...",ТР_БНУ_21.01.06-001.docx
7,В каком документе описаны доработки документа ...,Доработки по списанию НДС описаны в документ Т...,ТР_БНУ_21.01.10-001.docx
8,Какие доработки по документу Поступление и рег...,При проведении документа если есть строки со с...,ТР_БНУ_21.04.04-001.docx
9,В какие документы необходимо добавить реквизит...,"В документы ""Корректировка поступления"" и ""Кор...",ТР_БНУ_21.12.01-001.docx


In [247]:
# promt_keyword = """Контекст: мы выполняем работы по архивации документов, 
# необходимо индексировать документы для последующего поиска информации по ним
# Роль: твоя роль находить в документе ключевые термины, чтобы по ним в последющем осуществлять поиск
# Задача: Тебе предоставлен документ {doc_for_context} ты должен извлечь из предоставленного документ ключевые термины 
# Критерии Качества: номер законодательного акта, наименование законодательного акта, дата принятия, орган который принял законодательный акт, редакция и прочее.
# необходимо предоставить слова через запятую, без комментариев, на русском языке"""

#llm_litle = OllamaLLM(
#    model="deepseek-r1:1.5b", callback_manager=CallbackManager([StreamingStdOutCallbackHandler()]), temperature = "0.0"
#)
llm_litle = OllamaLLM(model="llama3:latest", temperature = "0.0")
promt_keyword = """Вы полезный ассистент. Вы отвечаете на вопросы о документации, используя эти данные: {doc_for_context}. Ответь на русском языке на этот запрос: какое наименование документа. Укажи только ответ, без комментариев"""
#promt_keyword = "Выдели 10 ключевых слов или фраз из текста {doc_for_context}. Они должны: Помогать в поиске этого документа. Включать название, номер, дату, имена, организации и другие важные обязательства.Быть конкретными, без общих слов. Ответ дай на русском языке. Формат ответа: список из 10 пунктов"
# Текст для анализа:
#promt_keyword = "Составь ТОП-5 вопросов и кратко ответь на них для блока FAQ для веб-страницы по тексту {doc_for_context} Ответ дай на русском языке"
# {doc_for_context}"""

promt_keyword = promt_keyword.replace("{doc_for_context}", doc_char, 1)
llm_litle.invoke(promt_keyword)

'Постановление Правительства Москвы от 21 декабря 2021 года N 2152-ПП (ред. от 30 июля 2024) "Об утверждении нормативов градостроительного проектирования города Москвы в области торговли, обслуживания и обеспечения комфортных и благоприятных условий проживания граждан в объектах жилого назначения".'

In [33]:
promt_category = {
    "Закон или нормативный акт": "Tom", 
    "Договор": "Bob", 
    "Письмо": "Bill",
    "Глоссарий": "Bill",
    "Практическая, курсовая или дипломная работа": "Bill",
    "Перечень формулировок и ответов": "Bill",
    "Информация о предприятии": "Bill",
    "Техническое задание или технический проект": "Bill",
    "Рассказ, роман, повесть": "Bill",
}

category_str = ""

for category, promt in promt_category.items():
    category_str = category_str + category + "\n"
#loader = PyPDFLoader("/Users/alexeyvaganov/doc/Работа/КЛУБ РАЗРАБОТЧИКОВ/file_for_RAG/dogovor-na-okazanie-uslug-universalnyj.pdf")
#loader = PyPDFLoader("The_Daughter_of_The_Commandant.pdf")
loader = PyPDFLoader("/Users/alexeyvaganov/doc/Работа/КЛУБ РАЗРАБОТЧИКОВ/Файлы/ППМ_от_21_12_2021_N_2152_ПП.pdf")

documents = loader.load()
doc_for_context = documents[0:1000]
doc_for_context = doc_for_context.replace("\n", "")
#promt_category = f"Тебе предоставлен текст {doc_for_context} ты должен определить к какой категории он относится, вот список категорий {category_str} укажи категорию которая больше всего подходит к тексту."
promt_category = f"Тебе предоставлен текст {doc_for_context} ты должен определить к какой категории он относится, вот список категорий {category_str} укажи категорию которая больше всего подходит к тексту."
llm.invoke(promt_category)

AttributeError: 'list' object has no attribute 'replace'

In [31]:
promt_category

'Тебе предоставлен текст [Document(metadata={\'source\': \'/Users/alexeyvaganov/doc/Работа/КЛУБ РАЗРАБОТЧИКОВ/Файлы/ППМ_от_21_12_2021_N_2152_ПП.pdf\', \'page\': 0}, page_content=\' \\nПостановление Правительства Москвы от \\n21.12.2021 N 2152-ПП \\n(ред. от 30.07.2024) \\n"Об утверждении нормативов \\nградостроительного проектирования города \\nМосквы в области торговли, обслуживания \\nи обеспечения комфортных и \\nблагоприятных условий проживания \\nграждан в объектах жилого назначения" \\nДокумент предоставлен КонсультантПлюс \\n \\nwww.consultant.ru \\n \\nДата сохранения: 29.08.2024 \\n  \\n\'), Document(metadata={\'source\': \'/Users/alexeyvaganov/doc/Работа/КЛУБ РАЗРАБОТЧИКОВ/Файлы/ППМ_от_21_12_2021_N_2152_ПП.pdf\', \'page\': 1}, page_content=\'Постановление Правительства Москвы от 21.12.2021 N 2152-ПП \\n(ред. от 30.07.2024) \\n"Об утверждении нормативов градостроител... \\nДокумент предоставлен КонсультантПлюс \\nДата сохранения: 29.08.2024 \\n \\n  \\n \\nКонсультантПлюс \\

In [9]:
#loader = PyPDFLoader("/Users/alexeyvaganov/doc/Работа/КЛУБ РАЗРАБОТЧИКОВ/file_for_RAG/dogovor-na-okazanie-uslug-universalnyj.pdf")
#loader = PyPDFLoader("The_Daughter_of_The_Commandant.pdf")
loader = PyPDFLoader("/Users/alexeyvaganov/doc/Работа/КЛУБ РАЗРАБОТЧИКОВ/Файлы/ППМ_от_21_12_2021_N_2152_ПП.pdf")

documents = loader.load()
doc_for_context = documents[0:1000]
#question = f"Ты рубрикатор. Тебе необходимо извлекать ключевые слова из текста для последующего поиска. Например, это могут быть название документа, навание разделов, номер и дата договора, автор и прочее. Извлеки десять ключевых слова из текста {doc_for_context}. Ответь на русском языке. Ответ представь только из этих слов разделенных запятой "
#question = f"Ты архивариус рубрикатор. Тебе необходимо находить важную поисковую информацию в тексте, чтобы потом осуществлять по ней поиск. Например, это могут быть название, навание разделов, номер и дата договора, автор и прочее. Извлеки важные поисковые индикатора из текста {documents}. Ответь на русском языке."
#question = f"Ты  рубрикатор. Тебе необходимо извлекать ключевые слова из текста для последующего поиска. Например, это могут быть название документа, автор документа, номер и дата договора и прочее. Извлеки десять ключевых слова из текста {documents}. Ответь на русском языке."

question = f" Выдели 10 ключевых слов или фраз из текста {documents}. Они должны: Помогать в поиске этого документа. Включать название, номер, дату, имена, организации, предмет договора, сумму и важные обязательства.Быть конкретными, без общих слов. Ответ дай на русском языке. Формат ответа: список из 10 пунктов.Пример:Договор на оказание услуг№ 1234525 сентября 2024ООО 'Рога и копыта'"

llm.invoke(question)

Вот 10 ключевых слов или фраз, которые помогут в поиске документа:

1. Постановление Правительства Москвы
2. Нормативы градостроительства
3. Территориальное планирование
4. Планировка территории
5. Общественный транспорт
6. Торговля и обслуживание
7. Жилые территории
8. Максимально допустимый уровень территориальной доступности
9. Минимально допустимый уровень обеспеченности населения объектами торговли и обслуживания
10. НГП ОЖ (Нормативы градостроительства и планирования территории в городе Москве)

Эти слова и фразы помогут вам найти документ, который является предметом поиска.

'Вот 10 ключевых слов или фраз, которые помогут в поиске документа:\n\n1. Постановление Правительства Москвы\n2. Нормативы градостроительства\n3. Территориальное планирование\n4. Планировка территории\n5. Общественный транспорт\n6. Торговля и обслуживание\n7. Жилые территории\n8. Максимально допустимый уровень территориальной доступности\n9. Минимально допустимый уровень обеспеченности населения объектами торговли и обслуживания\n10. НГП ОЖ (Нормативы градостроительства и планирования территории в городе Москве)\n\nЭти слова и фразы помогут вам найти документ, который является предметом поиска.'

In [24]:
documents[0].metadata["content"] = "капитанская дочка"

In [25]:
print(type(documents))
print(type(documents[0]))
print("-----")
print(documents[0].metadata)

<class 'list'>
<class 'langchain_core.documents.base.Document'>
-----
{'source': 'The_Daughter_of_The_Commandant.pdf', 'page': 0, 'content': 'капитанская дочка'}


In [5]:
# Если у вас нет видеокарты, укажите 'device': 'cpu'
# Выбираем эмбединг (чтобы можно было перевести текст в вектор)
hf_embeddings_model = HuggingFaceEmbeddings(
    model_name="cointegrated/LaBSE-en-ru", model_kwargs={"device": "cpu"}
)

/Users/alexeyvaganov/miniforge3/lib/python3.9/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/Users/alexeyvaganov/miniforge3/lib/python3.9/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [6]:
#  Загрузка в базу данных
db = Chroma.from_documents(
    documents,
    hf_embeddings_model,
    client_settings=Settings(anonymized_telemetry=False),
)

In [7]:
prompt = "Кому подарили заячий тулупчик?"

In [62]:
prompt = "Кем оказался вожатый?"

In [8]:
data = db.similarity_search(prompt,k=4)
question = f"Используя эти данные: {data}. Ответь на русском языке на этот запрос: {prompt} и укажи source "
llm.invoke([HumanMessage(content=question)])

В тексте не указано, кому именно подарили заячий тулупчик. Однако можно предположить, что это Гринев подарил вожатому, о чем сообщается в документе с метаданными {'page': 0, 'source': 'The_Daughter_of_The_Commandant.pdf'} и содержимым 'чутья» которого поражают Гринева, человек просит проводить их до ближайшего жилья. \nВ кибитке Гриневу снится сон, будто он приезжает в усадьбу, застает отца при смерти...'.

"В тексте не указано, кому именно подарили заячий тулупчик. Однако можно предположить, что это Гринев подарил вожатому, о чем сообщается в документе с метаданными {'page': 0, 'source': 'The_Daughter_of_The_Commandant.pdf'} и содержимым 'чутья» которого поражают Гринева, человек просит проводить их до ближайшего жилья. \\nВ кибитке Гриневу снится сон, будто он приезжает в усадьбу, застает отца при смерти...'."

In [56]:
print(type(data))
print(f"Total documents: {len(data)}")
print(data[0])
print(data[1])
print(data[2])
print(data[3])


<class 'list'>
Total documents: 4
page_content='но силы  неравны . Захватив  крепость , Пугачев  устраивает  «суд». 
Ивана  Кузьмича  и его товарищей  казнят  (вешают ). Когда  очередь  доходит  до Гринева , 
Савельич  бросается  в ноги  Пугачеву , умоляя  пощадить  «барского  дитятю », обещает  
выкуп . Пугачев  соглашается . Жители  города  и гарнизонные  солдаты  присягают  Пугачеву . 
На крыльцо  выводят  раздетую  Василису  Егоровну , убивают  ее. Пугачев  уезжает . 
Глава  8 
Незваный  гость  
Гринева  мучает  мысль  о судьбе  Маши . Ее прячет  у себя попадья , от которой  Гринев  
узнает , что Швабрин  перешел  на сторону  Пугачева . Савельич  сообщает  Гриневу , что 
узнал  в Пугачеве  вожатого . Пугачев  зовет  Гринева  к себе. Гринев  отправляется . «Все 
обходились  между  собою  как товарищи  и не оказывали  никакого  особенного  предпочте -
ния своему  предводителю ... Каждый  хвастал , предлагал  свои  мнения  и свободно  оспори -' metadata={'page': 1, 'source': 'The_Daug

In [58]:
print(question)

Используя эти данные: [Document(metadata={'page': 1, 'source': 'The_Daughter_of_The_Commandant.pdf'}, page_content='но силы  неравны . Захватив  крепость , Пугачев  устраивает  «суд». \nИвана  Кузьмича  и его товарищей  казнят  (вешают ). Когда  очередь  доходит  до Гринева , \nСавельич  бросается  в ноги  Пугачеву , умоляя  пощадить  «барского  дитятю », обещает  \nвыкуп . Пугачев  соглашается . Жители  города  и гарнизонные  солдаты  присягают  Пугачеву . \nНа крыльцо  выводят  раздетую  Василису  Егоровну , убивают  ее. Пугачев  уезжает . \nГлава  8 \nНезваный  гость  \nГринева  мучает  мысль  о судьбе  Маши . Ее прячет  у себя попадья , от которой  Гринев  \nузнает , что Швабрин  перешел  на сторону  Пугачева . Савельич  сообщает  Гриневу , что \nузнал  в Пугачеве  вожатого . Пугачев  зовет  Гринева  к себе. Гринев  отправляется . «Все \nобходились  между  собою  как товарищи  и не оказывали  никакого  особенного  предпочте -\nния своему  предводителю ... Каждый  хвастал , предлага

# УСТАНОВКА ПАКЕТОВ

In [7]:
pip install langchain_huggingface

  Using cached langchain_huggingface-0.1.2-py3-none-any.whl (21 kB)
  Using cached huggingface_hub-0.26.2-py3-none-any.whl (447 kB)


  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.22.2
    Uninstalling huggingface-hub-0.22.2:
      Successfully uninstalled huggingface-hub-0.22.2
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install langchain_ollama

  Using cached langchain_ollama-0.2.0-py3-none-any.whl (14 kB)
  Using cached ollama-0.3.3-py3-none-any.whl (10 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.3/409.3 KB 896.3 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 307.0/307.0 KB 731.7 kB/s eta 0:00:00a 0:00:01
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl (54 kB)
  Attempting uninstall: ollama
    Found existing installation: ollama 0.2.0
    Uninstalling ollama-0.2.0:
      Successfully uninstalled ollama-0.2.0
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.1.54
    Uninstalling langsmith-0.1.54:
      Successfully uninstalled langsmith-0.1.54
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.1.50
    Uninstalling langchain-core-0.1.50:
      Successfully uninstalled langchain-core-0.1.50
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the s

In [1]:
pip install langchain-huggingface

Note: you may need to restart the kernel to use updated packages.


In [10]:
pip install sentence-transformers

     |████████████████████████████████| 268 kB 411 kB/s eta 0:00:01
     |████████████████████████████████| 150.8 MB 525 kB/s eta 0:00:01    |██▋                             | 12.1 MB 926 kB/s eta 0:02:30     |███▏                            | 15.1 MB 793 kB/s eta 0:02:51 MB 812 kB/s eta 0:01:33
     |████████████████████████████████| 10.0 MB 505 kB/s eta 0:00:01
     |████████████████████████████████| 447 kB 615 kB/s eta 0:00:01
     |████████████████████████████████| 2.7 MB 306 kB/s eta 0:00:01
     |████████████████████████████████| 17.7 MB 420 kB/s eta 0:00:01     |██████████████████████████      | 14.4 MB 547 kB/s eta 0:00:06
  Attempting uninstall: numpy
    Found existing installation: numpy 1.24.2
    Uninstalling numpy-1.24.2:
      Successfully uninstalled numpy-1.24.2
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.22.2
    Uninstalling huggingface-hub-0.22.2:
      Successfully uninstalled huggingface-hub-0.22.2
  Attempting uninst

In [8]:
pip install pypdf

     |████████████████████████████████| 297 kB 682 kB/s eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [6]:
pip install chromadb

     |████████████████████████████████| 615 kB 922 kB/s eta 0:00:01
     |████████████████████████████████| 118 kB 10.6 MB/s eta 0:00:01
     |████████████████████████████████| 7.1 MB 624 kB/s eta 0:00:011
     |████████████████████████████████| 242 kB 3.9 MB/s eta 0:00:01
     |████████████████████████████████| 78 kB 5.5 MB/s eta 0:00:011
  Using cached overrides-7.7.0-py3-none-any.whl (17 kB)
     |████████████████████████████████| 1.9 MB 127 kB/s eta 0:00:01
     |████████████████████████████████| 11.0 MB 404 kB/s eta 0:00:01
  Using cached PyPika-0.48.9-py2.py3-none-any.whl
     |████████████████████████████████| 54 kB 1.0 MB/s eta 0:00:01
     |████████████████████████████████| 63 kB 695 kB/s eta 0:00:01
     |████████████████████████████████| 94 kB 806 kB/s eta 0:00:011
     |████████████████████████████████| 44 kB 877 kB/s eta 0:00:011
     |████████████████████████████████| 195 kB 927 kB/s eta 0:00:01
     |████████████████████████████████| 472 kB 936 kB/s eta 0:00:01
     |███

  Using cached coloredlogs-15.0.1-py2.py3-none-any.whl (46 kB)
  Using cached Deprecated-1.2.14-py2.py3-none-any.whl (9.6 kB)
     |████████████████████████████████| 55 kB 1.8 MB/s eta 0:00:011
     |████████████████████████████████| 414 kB 1.1 MB/s eta 0:00:01
     |████████████████████████████████| 159 kB 1.1 MB/s eta 0:00:01
  Using cached asgiref-3.8.1-py3-none-any.whl (23 kB)
  Using cached monotonic-1.6-py2.py3-none-any.whl (8.2 kB)
  Using cached backoff-2.2.1-py3-none-any.whl (15 kB)
  Using cached markdown_it_py-3.0.0-py3-none-any.whl (87 kB)
  Using cached pygments-2.18.0-py3-none-any.whl (1.2 MB)
  Using cached mdurl-0.1.2-py3-none-any.whl (10.0 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl (9.8 kB)
     |████████████████████████████████| 201 kB 488 kB/s eta 0:00:01
     |████████████████████████████████| 375 kB 978 kB/s eta 0:00:01
     |████████████████████████████████| 800 kB 1.2 MB/s eta 0:00:01
     |████████████████████████████████| 159 kB 793 kB/s eta 0:00

Note: you may need to restart the kernel to use updated packages.
